# device_messages_raw
This table stores raw, timestamped sensor readings emitted by each device, keyed by device_id. It contains the core measurement fields (like sensor_type and distance) exactly as they arrived from the device, so it’s the source of truth for the time-series signal. It helps with ML features because it’s where we compute cleaned numeric values and rolling/windowed features over time (deltas, averages, variance, etc.).


# rapid_step_tests_raw
This table stores step-test sessions for each device, including the start_time/stop_time window and the step_points array describing step timing within the test. It’s used as ground truth context to align the raw sensor stream with known step activity. It helps with ML features by providing labels (step vs no_step, and potentially cadence/step timing targets) so we can train and validate models using the sensor signal.

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.device_messages_raw;

In [0]:
%sql
SELECT *
FROM workspace.bronze.device_messages_raw
LIMIT 20;

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.rapid_step_tests_raw;

In [0]:
%sql
SELECT *
FROM workspace.bronze.rapid_step_tests_raw
LIMIT 20;

In [0]:
%sql
SELECT
typeof(step_points) AS step_points_type,
typeof(step_points[0]) AS first_elem_type
FROM workspace.bronze.rapid_step_tests_raw
WHERE step_points IS NOT NULL
LIMIT 20;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.labeled_step_test (
timestamp LONG COMMENT 'Exact time when the sensor reading was recorded (matches device_messages_raw timestamp).',
device_id STRING COMMENT 'Device that produced the sensor reading.',
bronze_record_id LONG COMMENT 'Optional surrogate key from the bronze device_messages_raw row for lineage/debugging.',
test_id STRING COMMENT 'Optional id generated during ETL to group all rows belonging to the same rapid step test session.',

sensor_type STRING COMMENT 'Type of sensor used (example: ultrasonic_sensor). From device_messages_raw.',
distance_cm INT COMMENT 'Numeric distance in centimeters (converted from the raw distance field).',

step_label STRING COMMENT 'Label created by checking whether timestamp is inside a rapid step test window: step or no_step.',
source STRING COMMENT 'Indicates original source or tagging origin (example: device_messages_raw vs rapid_step_tests_raw).',

start_time LONG COMMENT 'Start time of the rapid step test window.',
stop_time LONG COMMENT 'End time of the rapid step test window.'
)
USING DELTA;

In [0]:
%sql
SELECT
  column_name AS `Column Name`,
  data_type AS `Data Type`,
  comment AS `Comment`
FROM workspace.information_schema.columns
WHERE table_schema = 'silver'
  AND table_name = 'labeled_step_test'
ORDER BY ordinal_position;